# ==================================================
# MULTI-ASSET PERFORMANCE ANALYTICS DASHBOARD
# ==================================================

SETUP & IMPORTS

In [28]:
!pip install -q yfinance plotly ipywidgets --upgrade

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

CONFIGURATION

In [29]:
ASSET_UNIVERSE = {
    "AAPL":     "Equity",
    "MSFT":     "Equity",
    "SPY":      "ETF",
    "QQQ":      "ETF",
    "GLD":      "Commodity",   # Gold ETP proxy (use "GC=F" for the futures contract)
    "USO":      "Commodity",   # Oil ETP proxy (use "CL=F" for the futures contract)
    "EURUSD=X": "Currency",
    "BTC-USD":  "Crypto",
    "ETH-USD":  "Crypto",
}

START_DATE = "2021-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")

RISK_FREE_RATE   = 0.04     # annualized, e.g. 4% T-bill proxy
TRADING_DAYS     = 252
ROLLING_WINDOW   = 63       # ~1 trading quarter, for rolling Sharpe
MAX_FFILL_DAYS   = 5        # cap on forward-fill for short data gaps

ASSET SELECTION WIDGET (INTERACTIVE)

In [30]:
import ipywidgets as widgets
from IPython.display import display

ticker_select = widgets.SelectMultiple(
    options=list(ASSET_UNIVERSE.keys()),
    value=list(ASSET_UNIVERSE.keys()),
    description="Assets:",
    layout=widgets.Layout(width="300px", height="160px"),
)
start_picker = widgets.DatePicker(description="Start", value=pd.to_datetime(START_DATE))
end_picker   = widgets.DatePicker(description="End",   value=pd.to_datetime(END_DATE))

def apply_selection(_):
    global ASSET_UNIVERSE, START_DATE, END_DATE
    chosen = list(ticker_select.value)
    if not chosen:
        print("Select at least one asset.")
        return
    ASSET_UNIVERSE = {t: ASSET_UNIVERSE[t] for t in chosen}
    START_DATE = start_picker.value.strftime("%Y-%m-%d")
    END_DATE   = end_picker.value.strftime("%Y-%m-%d")
    print(f"Active universe: {chosen}\nRange: {START_DATE} -> {END_DATE}")
    print("Now re-run Cells 4-10.")

apply_btn = widgets.Button(description="Apply Selection", button_style="primary")
apply_btn.on_click(apply_selection)

display(widgets.VBox([ticker_select, widgets.HBox([start_picker, end_picker]), apply_btn]))

DATA COLLECTION PIPELINE

In [31]:
def download_prices(tickers: dict, start: str, end: str,
                     retries: int = 3, base_delay: float = 1.5) -> pd.DataFrame:
    """
    Downloads adjusted close prices per ticker individually so a single bad
    ticker (delisted, typo, API hiccup) doesn't fail the whole batch.

    Retries each ticker with exponential backoff — the most common cause of
    "all tickers failed" is Yahoo Finance rate-limiting a shared Colab IP,
    not a bad ticker list. Failure reasons are always printed BEFORE any
    exception is raised, so the real per-ticker error is never hidden.

    Returns a wide DataFrame: index=date, columns=ticker.
    """
    import time

    series_list, failed = [], []

    for ticker in tickers:
        last_err = None
        for attempt in range(1, retries + 1):
            try:
                data = yf.download(
                    ticker, start=start, end=end,
                    auto_adjust=True, progress=False, threads=False,
                )
                if data.empty:
                    raise ValueError("empty response")
                # yfinance versions differ: some return flat columns
                # (data["Close"] -> Series), others return MultiIndex
                # columns [Price, Ticker] even for a single symbol
                # (data["Close"] -> 1-col DataFrame). Normalize to a Series
                # either way so .rename() reliably sets the series name.
                close = data["Close"]
                if isinstance(close, pd.DataFrame):
                    close = close.iloc[:, 0]
                series_list.append(close.rename(ticker))
                last_err = None
                break
            except Exception as e:
                last_err = e
                time.sleep(base_delay * attempt)  # backoff before retry
        if last_err is not None:
            failed.append((ticker, repr(last_err)))
        time.sleep(base_delay)  # pace requests between tickers, avoid rate-limit

    if failed:
        print("WARNING — failed downloads (excluded from dashboard):")
        for t, err in failed:
            print(f"  {t}: {err}")

    if not series_list:
        raise RuntimeError(
            "All ticker downloads failed (see per-ticker errors printed above). "
            "Most likely causes, in order of probability:\n"
            "  1. Yahoo Finance rate-limited this Colab runtime's shared IP — "
            "wait a few minutes and rerun, or run on a different network.\n"
            "  2. yfinance is outdated — rerun Cell 1: `pip install -U yfinance`, "
            "then Runtime > Restart Runtime, then rerun from Cell 1.\n"
            "  3. No internet egress from this runtime/firewall blocking Yahoo.\n"
            "Run the single-ticker diagnostic snippet (see chat) to see the raw "
            "underlying exception."
        )

    return pd.concat(series_list, axis=1)


raw_prices = download_prices(ASSET_UNIVERSE, START_DATE, END_DATE)
print(f"Downloaded shape: {raw_prices.shape}")
raw_prices.tail()


Downloaded shape: (2005, 9)


,AAPL,MSFT,SPY,QQQ,GLD,USO,EURUSD=X,BTC-USD,ETH-USD
Date,,,,,,,,,
2026-06-24,293.0800,365.4600,733.2400,710.6200,365.9200,106.2900,1.1380,"60,995.1328","1,619.9249"
2026-06-25,275.1500,352.8300,734.3000,716.3800,369.4600,109.3100,1.1354,"59,721.6758","1,564.8167"
2026-06-26,283.7800,372.9700,728.9900,706.5200,373.6300,105.4800,1.1362,"60,016.4297","1,576.6161"
2026-06-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"59,940.0977","1,571.5883"
2026-06-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"59,532.3398","1,570.3612"


DATA CLEANING & ALIGNMENT

In [32]:
def clean_price_data(df: pd.DataFrame, max_ffill: int, min_coverage: float = 0.90) -> pd.DataFrame:
    """
    - Forward-fills gaps up to `max_ffill` days (handles holiday/calendar
      mismatches between e.g. equities (5d/week) and crypto (7d/week)).
    - Drops any column whose post-fill missing-data ratio exceeds
      (1 - min_coverage) — treated as delisted/unreliable rather than faked.
    - Trims to the first date where ALL remaining assets have data, so every
      asset's cumulative return starts from the same baseline.
    """
    filled = df.ffill(limit=max_ffill)

    coverage = filled.notna().mean()
    keep_cols = coverage[coverage >= min_coverage].index.tolist()
    dropped = set(df.columns) - set(keep_cols)
    if dropped:
        print(f"Dropping (insufficient coverage): {sorted(dropped)}")

    filled = filled[keep_cols]
    first_valid_idx = filled.dropna().index.min()
    aligned = filled.loc[first_valid_idx:].dropna()

    if aligned.shape[1] < 1:
        raise RuntimeError("No assets survived cleaning — widen date range or check tickers.")

    return aligned


clean_prices = clean_price_data(raw_prices, MAX_FFILL_DAYS)
ACTIVE_TICKERS = clean_prices.columns.tolist()
print(f"Clean shape: {clean_prices.shape} | Active tickers: {ACTIVE_TICKERS}")
print(f"Effective start date: {clean_prices.index.min().date()}")

Clean shape: (2002, 9) | Active tickers: ['AAPL', 'MSFT', 'SPY', 'QQQ', 'GLD', 'USO', 'EURUSD=X', 'BTC-USD', 'ETH-USD']
Effective start date: 2021-01-04


RETURNS ENGINE

In [33]:
def compute_daily_returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().dropna(how="all")

def compute_cumulative_returns(daily_returns: pd.DataFrame) -> pd.DataFrame:
    """Geometric cumulative return, indexed to start at 0 (i.e. 1.0 = 100%)."""
    return (1.0 + daily_returns).cumprod() - 1.0

def normalize_to_100(prices: pd.DataFrame) -> pd.DataFrame:
    """For the equity-curve chart: every asset starts at 100."""
    return prices / prices.iloc[0] * 100.0


daily_returns      = compute_daily_returns(clean_prices)
cumulative_returns = compute_cumulative_returns(daily_returns)
indexed_prices     = normalize_to_100(clean_prices)


RISK & PERFORMANCE METRICS

In [34]:
def annualized_return(daily_returns: pd.Series, periods: int = TRADING_DAYS) -> float:
    total_return = (1.0 + daily_returns).prod() - 1.0
    n = daily_returns.count()
    if n == 0:
        return np.nan
    return (1.0 + total_return) ** (periods / n) - 1.0

def annualized_volatility(daily_returns: pd.Series, periods: int = TRADING_DAYS) -> float:
    return daily_returns.std() * np.sqrt(periods)

def sharpe_ratio(daily_returns: pd.Series, rf: float = RISK_FREE_RATE,
                  periods: int = TRADING_DAYS) -> float:
    vol = annualized_volatility(daily_returns, periods)
    if vol == 0 or np.isnan(vol):
        return np.nan
    return (annualized_return(daily_returns, periods) - rf) / vol

def sortino_ratio(daily_returns: pd.Series, rf: float = RISK_FREE_RATE,
                   periods: int = TRADING_DAYS) -> float:
    downside = daily_returns[daily_returns < 0]
    if downside.empty:
        return np.nan  # no downside observed — ratio undefined, not infinite
    downside_dev = np.sqrt((downside ** 2).mean()) * np.sqrt(periods)
    if downside_dev == 0:
        return np.nan
    return (annualized_return(daily_returns, periods) - rf) / downside_dev

def max_drawdown(cum_returns: pd.Series) -> float:
    wealth_index = 1.0 + cum_returns
    running_max = wealth_index.cummax()
    drawdown = wealth_index / running_max - 1.0
    return drawdown.min()

def drawdown_series(cum_returns: pd.Series) -> pd.Series:
    wealth_index = 1.0 + cum_returns
    running_max = wealth_index.cummax()
    return wealth_index / running_max - 1.0

def calmar_ratio(daily_returns: pd.Series, cum_returns: pd.Series,
                  periods: int = TRADING_DAYS) -> float:
    mdd = max_drawdown(cum_returns)
    if mdd == 0:
        return np.nan
    return annualized_return(daily_returns, periods) / abs(mdd)

def rolling_sharpe(daily_returns: pd.Series, window: int = ROLLING_WINDOW,
                    rf: float = RISK_FREE_RATE, periods: int = TRADING_DAYS) -> pd.Series:
    roll_mean = daily_returns.rolling(window).mean() * periods
    roll_std  = daily_returns.rolling(window).std() * np.sqrt(periods)
    return (roll_mean - rf) / roll_std.replace(0, np.nan)


def build_metrics_table(daily_returns: pd.DataFrame, cum_returns: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ticker in daily_returns.columns:
        r, c = daily_returns[ticker].dropna(), cum_returns[ticker].dropna()
        rows.append({
            "Ticker":            ticker,
            "Asset Class":       ASSET_UNIVERSE.get(ticker, "N/A"),
            "Total Return":      c.iloc[-1] if len(c) else np.nan,
            "CAGR":              annualized_return(r),
            "Ann. Volatility":   annualized_volatility(r),
            "Sharpe":            sharpe_ratio(r),
            "Sortino":           sortino_ratio(r),
            "Max Drawdown":      max_drawdown(c),
            "Calmar":            calmar_ratio(r, c),
            "Skew":              r.skew(),
            "Kurtosis":          r.kurtosis(),
        })
    return pd.DataFrame(rows).set_index("Ticker").sort_values("Sharpe", ascending=False)


metrics_table = build_metrics_table(daily_returns, cumulative_returns)
drawdowns     = cumulative_returns.apply(lambda col: drawdown_series(col.dropna()))
rolling_sharpe_df = daily_returns.apply(rolling_sharpe)

display(metrics_table.style.format({
    "Total Return": "{:.2%}", "CAGR": "{:.2%}", "Ann. Volatility": "{:.2%}",
    "Sharpe": "{:.2f}", "Sortino": "{:.2f}", "Max Drawdown": "{:.2%}",
    "Calmar": "{:.2f}", "Skew": "{:.2f}", "Kurtosis": "{:.2f}",
}))

,Asset Class,Total Return,CAGR,Ann. Volatility,Sharpe,Sortino,Max Drawdown,Calmar,Skew,Kurtosis
Ticker,,,,,,,,,,
SPY,ETF,112.88%,9.98%,13.98%,0.43,0.35,-24.50%,0.41,0.40,14.30
USO,Commodity,225.76%,16.04%,29.86%,0.40,0.32,-36.23%,0.44,-0.18,5.69
QQQ,ETF,135.93%,11.42%,18.68%,0.40,0.32,-35.12%,0.33,0.23,8.51
GLD,Commodity,104.92%,9.46%,14.96%,0.36,0.29,-26.21%,0.36,-0.80,11.84
AAPL,Equity,125.69%,10.79%,22.95%,0.30,0.25,-33.36%,0.32,0.52,10.45
MSFT,Equity,79.35%,7.63%,22.04%,0.16,0.14,-37.15%,0.21,0.10,6.43
BTC-USD,Crypto,86.20%,8.14%,47.86%,0.09,0.09,-76.63%,0.11,0.12,3.94
ETH-USD,Crypto,50.96%,5.32%,63.93%,0.02,0.02,-79.35%,0.07,0.11,4.46
EURUSD=X,Currency,-7.26%,-0.94%,6.25%,-0.79,-0.69,-22.24%,-0.04,0.26,4.03


CORRELATION ENGINE

In [35]:
def compute_correlation_matrix(daily_returns: pd.DataFrame) -> pd.DataFrame:
    if daily_returns.shape[1] < 2:
        print("Need at least 2 assets for a correlation matrix.")
        return pd.DataFrame()
    return daily_returns.corr()


correlation_matrix = compute_correlation_matrix(daily_returns)
correlation_matrix

,AAPL,MSFT,SPY,QQQ,GLD,USO,EURUSD=X,BTC-USD,ETH-USD
AAPL,1.0000,0.5834,0.7370,0.7482,0.0664,0.0382,-0.0407,0.2243,0.2413
MSFT,0.5834,1.0000,0.7108,0.7572,0.0818,0.0336,0.0017,0.2950,0.2906
SPY,0.7370,0.7108,1.0000,0.9455,0.1563,0.1093,-0.0080,0.3582,0.3696
QQQ,0.7482,0.7572,0.9455,1.0000,0.1526,0.0559,-0.0116,0.3602,0.3714
GLD,0.0664,0.0818,0.1563,0.1526,1.0000,0.1243,0.0090,0.0889,0.0912
USO,0.0382,0.0336,0.1093,0.0559,0.1243,1.0000,-0.0466,0.0334,0.0512
EURUSD=X,-0.0407,0.0017,-0.0080,-0.0116,0.0090,-0.0466,1.0000,0.0081,-0.0135
BTC-USD,0.2243,0.2950,0.3582,0.3602,0.0889,0.0334,0.0081,1.0000,0.8211
ETH-USD,0.2413,0.2906,0.3696,0.3714,0.0912,0.0512,-0.0135,0.8211,1.0000


VISUALIZATION FUNCTIONS

In [36]:
PLOTLY_TEMPLATE = "plotly_white"

def plot_equity_curve(indexed_prices: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    for ticker in indexed_prices.columns:
        fig.add_trace(go.Scatter(
            x=indexed_prices.index, y=indexed_prices[ticker],
            name=f"{ticker} ({ASSET_UNIVERSE.get(ticker,'')})", mode="lines",
        ))
    fig.update_layout(
        title="Equity Curve — Normalized to 100", xaxis_title="Date",
        yaxis_title="Indexed Value (Start = 100)", template=PLOTLY_TEMPLATE,
        legend_title="Asset", hovermode="x unified",
    )
    return fig

def plot_drawdown(drawdowns: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    for ticker in drawdowns.columns:
        fig.add_trace(go.Scatter(
            x=drawdowns.index, y=drawdowns[ticker] * 100,
            name=ticker, mode="lines", fill="tozeroy",
        ))
    fig.update_layout(
        title="Drawdown (Underwater Chart)", xaxis_title="Date",
        yaxis_title="Drawdown (%)", template=PLOTLY_TEMPLATE, hovermode="x unified",
    )
    return fig

def plot_rolling_sharpe(rolling_sharpe_df: pd.DataFrame, window: int) -> go.Figure:
    fig = go.Figure()
    for ticker in rolling_sharpe_df.columns:
        fig.add_trace(go.Scatter(
            x=rolling_sharpe_df.index, y=rolling_sharpe_df[ticker],
            name=ticker, mode="lines",
        ))
    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    fig.update_layout(
        title=f"Rolling {window}-Day Annualized Sharpe Ratio", xaxis_title="Date",
        yaxis_title="Sharpe Ratio", template=PLOTLY_TEMPLATE, hovermode="x unified",
    )
    return fig

def plot_correlation_heatmap(correlation_matrix: pd.DataFrame) -> go.Figure:
    if correlation_matrix.empty:
        return go.Figure()
    fig = px.imshow(
        correlation_matrix, text_auto=".2f", color_continuous_scale="RdBu_r",
        zmin=-1, zmax=1, aspect="auto",
        title="Return Correlation Matrix",
    )
    fig.update_layout(template=PLOTLY_TEMPLATE)
    return fig

def plot_risk_return_scatter(metrics_table: pd.DataFrame) -> go.Figure:
    fig = px.scatter(
        metrics_table.reset_index(), x="Ann. Volatility", y="CAGR",
        size=metrics_table["Sharpe"].clip(lower=0.01).reset_index(drop=True),
        color="Asset Class", text="Ticker", hover_name="Ticker",
        title="Risk-Return Profile (bubble size = Sharpe)",
    )
    fig.update_traces(textposition="top center")
    fig.update_layout(
        xaxis_title="Annualized Volatility", yaxis_title="Annualized Return (CAGR)",
        xaxis_tickformat=".1%", yaxis_tickformat=".1%", template=PLOTLY_TEMPLATE,
    )
    return fig

DASHBOARD ORCHESTRATION & EXECUTION

In [37]:
def run_dashboard():
    """Renders the full chart stack in sequence. Re-run after changing CONFIG."""
    plot_equity_curve(indexed_prices).show()
    plot_drawdown(drawdowns).show()
    plot_rolling_sharpe(rolling_sharpe_df, ROLLING_WINDOW).show()
    plot_correlation_heatmap(correlation_matrix).show()
    plot_risk_return_scatter(metrics_table).show()
    print("\n=== CONSOLIDATED METRICS ===")
    display(metrics_table)


run_dashboard()


=== CONSOLIDATED METRICS ===


,Asset Class,Total Return,CAGR,Ann. Volatility,Sharpe,Sortino,Max Drawdown,Calmar,Skew,Kurtosis
Ticker,,,,,,,,,,
SPY,ETF,1.1288,0.0998,0.1398,0.4278,0.3468,-0.2450,0.4075,0.3981,14.2996
USO,Commodity,2.2576,0.1604,0.2986,0.4031,0.3248,-0.3623,0.4426,-0.1831,5.6905
QQQ,ETF,1.3593,0.1142,0.1868,0.3970,0.3187,-0.3512,0.3251,0.2264,8.5079
GLD,Commodity,1.0492,0.0946,0.1496,0.3647,0.2893,-0.2621,0.3608,-0.7967,11.8441
AAPL,Equity,1.2569,0.1079,0.2295,0.2960,0.2478,-0.3336,0.3236,0.5187,10.4507
MSFT,Equity,0.7935,0.0763,0.2204,0.1649,0.1375,-0.3715,0.2055,0.0975,6.4273
BTC-USD,Crypto,0.8620,0.0814,0.4786,0.0866,0.0902,-0.7663,0.1063,0.1209,3.9406
ETH-USD,Crypto,0.5096,0.0532,0.6393,0.0207,0.0213,-0.7935,0.0671,0.1112,4.4620
EURUSD=X,Currency,-0.0726,-0.0094,0.0625,-0.7914,-0.6928,-0.2224,-0.0425,0.2562,4.0344
